In [43]:
import pandas as pd
import numpy as np
import os, math
from glob import glob
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
class Persona:
    def __init__(self, ruta_base, nombre, nodos, status):
        self.ruta_base = ruta_base
        self.nombre = nombre
        self.nodos = nodos
        self.recortes = [str(i) for i in range(1, 51)]
        self.status = status
        self.data = None

    def read_file(self, archivo):
        """Lee un archivo y lo retorna como DataFrame"""
        try:
            return pd.read_table(archivo, header=None)
        except Exception as e:
            print(f"Error leyendo {archivo}: {e}")
            return None

    def all_files(self):
        """Crea un DataFrame con columnas: nodo, recorte, indice, valor"""
        registros = []
        
        for nodo in self.nodos:
            for recorte in self.recortes:
                ruta = f"{self.ruta_base}{self.nombre}/{nodo}/recorte_{recorte}.txt"
                df = self.read_file(ruta)
                
                if df is not None and len(df) > 0:
                    for idx, row in df.iterrows():
                        try:
                            valor = float(row[0])  # Convertir a float
                            registros.append({
                                'nodo': nodo,
                                'recorte': int(recorte),
                                'indice': idx,
                                'valor': valor,
                                'persona': self.nombre,
                                'status': self.status
                            })
                        except (ValueError, TypeError):
                            continue  # Saltar valores no numéricos
        
        self.data = pd.DataFrame(registros)
        print(f"✓ {self.nombre}: {len(self.data)} registros cargados")
        return self.data

    def get_persona_df(self):
        return self.data

In [3]:
def cargar_grupo(lista_nombres, ruta_base, nodos, status, verbose=True):
    """
    Carga un grupo completo de personas
    
    Parámetros:
    - lista_nombres: lista de nombres de los sujetos
    - ruta_base: ruta base de los datos
    - nodos: lista de nodos
    - status: 0=control, 1=DCL
    - verbose: mostrar progreso
    
    Retorna:
    - Lista de objetos Persona
    """
    grupo = []
    for i, nombre in enumerate(lista_nombres):
        if verbose:
            print(f"Cargando {nombre} ({i+1}/{len(lista_nombres)})...")
        persona = Persona(ruta_base, nombre, nodos, status)
        persona.all_files()
        grupo.append(persona)
    return grupo

In [4]:
def estadistica_grupo(grupo_personas, nombre_grupo):
    """
    Calcula estadística descriptiva para un grupo de personas
    
    Parámetros:
    - grupo_personas: lista de objetos Persona
    - nombre_grupo: nombre del grupo (para imprimir)
    
    Retorna:
    - DataFrame con estadísticas combinadas
    """
    dfs = []
    for persona in grupo_personas:
        df = persona.get_persona_df()
        if df is not None and len(df) > 0:
            dfs.append(df)
    
    if not dfs:
        print(f"No hay datos para el grupo {nombre_grupo}")
        return None
    
    df_completo = pd.concat(dfs, ignore_index=True)
    
    print(f"\n{'='*50}")
    print(f"ESTADÍSTICA DESCRIPTIVA - GRUPO {nombre_grupo}")
    print(f"{'='*50}")
    print(f"Total de registros: {len(df_completo)}")
    print(f"Personas: {df_completo['persona'].nunique()}")
    print(f"Nodos: {df_completo['nodo'].nunique()}")
    print(f"Recortes: {df_completo['recorte'].nunique()}")
    
    print(f"\n--- Estadísticas generales del valor ---")
    print(df_completo['valor'].describe())
    
    print(f"\n--- Estadísticas por persona ---")
    stats_por_persona = df_completo.groupby('persona')['valor'].agg(['mean', 'std', 'count'])
    print(stats_por_persona)
    
    print(f"\n--- Estadísticas por nodo ---")
    stats_por_nodo = df_completo.groupby('nodo')['valor'].agg(['mean', 'std'])
    print(stats_por_nodo)
    
    return df_completo

In [5]:
def get_datos_nodo(grupo_personas, nodo):
    """Extrae todos los valores de un nodo específico de un grupo"""
    valores = []
    for persona in grupo_personas:
        df = persona.get_persona_df()
        if df is not None:
            nodo_data = df[df['nodo'] == nodo]['valor'].values
            valores.extend(nodo_data)
    return np.array(valores)

In [6]:
def calcular_sd1_sd2(valores):
    """
    Calcula SD1 y SD2 (métricas de Poincaré) para una serie de valores
    
    Parámetros:
    - valores: array o lista de valores (señal EEG)
    
    Retorna:
    - sd1, sd2: las dos métricas
    """
    if len(valores) < 2:
        return np.nan, np.nan
    diff = np.diff(valores)
    sd1 = np.std(diff) / np.sqrt(2)
    sd_points = np.std(valores)
    
    if len(valores) > 2:
        # Correlación de lag 1
        x = valores[:-1]
        y = valores[1:]
        corr = np.corrcoef(x, y)[0, 1]
        sd2 = np.sqrt(2 * sd_points**2 - (sd1**2))
    else:
        sd2 = np.nan
    
    return sd1, sd2

In [7]:
def calcular_sd1_sd2_por_grupo(df, groupby_cols):
    """
    Calcula SD1 y SD2 agrupando por columnas específicas
    
    Parámetros:
    - df: DataFrame con columna 'valor'
    - groupby_cols: lista de columnas para agrupar
    
    Retorna:
    - DataFrame con columnas: groupby_cols + ['sd1', 'sd2']
    """
    resultados = []
    
    for nombre_grupo, grupo in df.groupby(groupby_cols):
        if 'recorte' in grupo.columns and 'indice' in grupo.columns:
            grupo = grupo.sort_values(['recorte', 'indice'])
        
        valores = grupo['valor'].values
        sd1, sd2 = calcular_sd1_sd2(valores)
        
        # Crear diccionario con los resultados
        if isinstance(nombre_grupo, tuple):
            row = dict(zip(groupby_cols, nombre_grupo))
        else:
            row = {groupby_cols[0]: nombre_grupo}
        
        row['sd1'] = sd1
        row['sd2'] = sd2
        resultados.append(row)
    
    return pd.DataFrame(resultados)

In [8]:
def calcular_metricas_poincare_por_persona(grupo_personas):
    """
    Calcula SD1 y SD2 para cada persona en un grupo
    
    Parámetros:
    - grupo_personas: lista de objetos Persona
    
    Retorna:
    - DataFrame con columnas: persona, status, sd1, sd2
    """
    metricas = []
    
    for persona in grupo_personas:
        df = persona.get_persona_df()
        if df is not None and len(df) > 0:
            # Ordenar por recorte e índice para mantener secuencia temporal
            df_ordenado = df.sort_values(['recorte', 'indice'])
            sd1, sd2 = calcular_sd1_sd2(df_ordenado['valor'].values)
            
            metricas.append({
                'persona': persona.nombre,
                'status': persona.status,
                'sd1': sd1,
                'sd2': sd2
            })
    
    return pd.DataFrame(metricas)

In [9]:
def calcular_metricas_poincare_por_nodo(grupo_personas):
    """
    Calcula SD1 y SD2 para cada combinación persona-nodo
    
    Retorna DataFrame con: persona, nodo, status, sd1, sd2
    """
    metricas = []
    
    for persona in grupo_personas:
        df = persona.get_persona_df()
        if df is not None and len(df) > 0:
            for nodo in df['nodo'].unique():
                df_nodo = df[df['nodo'] == nodo].sort_values(['recorte', 'indice'])
                sd1, sd2 = calcular_sd1_sd2(df_nodo['valor'].values)
                
                metricas.append({
                    'persona': persona.nombre,
                    'nodo': nodo,
                    'status': persona.status,
                    'sd1': sd1,
                    'sd2': sd2
                })
    
    return pd.DataFrame(metricas)

In [10]:
def euclidean_distance(point1, point2):
    """Calcula la distancia euclidiana entre dos puntos"""
    return math.sqrt(sum((p1 - p2) ** 2 for p1, p2 in zip(point1, point2)))

def knn_predict(X_train, y_train, X_test, k=3):
    """
    Clasificador KNN usando distancia euclidiana
    
    Parámetros:
    X_train: lista de puntos de entrenamiento
    y_train: lista de etiquetas de entrenamiento
    X_test: lista de puntos a clasificar
    k: número de vecinos a considerar
    
    Retorna:
    lista de etiquetas predichas para X_test
    """
    predictions = []
    
    for test_point in X_test:
        # Calcular distancias a todos los puntos de entrenamiento
        distances = []
        for i, train_point in enumerate(X_train):
            dist = euclidean_distance(test_point, train_point)
            distances.append((dist, y_train[i]))
        
        # Ordenar por distancia y obtener los k vecinos más cercanos
        distances.sort(key=lambda x: x[0])
        k_neighbors = distances[:k]
        
        # Obtener las etiquetas de los k vecinos
        k_labels = [label for _, label in k_neighbors]
        
        # Votación por mayoría
        most_common = Counter(k_labels).most_common(1)[0][0]
        predictions.append(most_common)
    
    return predictions

In [11]:
def extraer_metricas_por_persona(grupo_personas):
    """
    Extrae SD1 y SD2 para cada persona de un grupo
    """
    metricas = []
    
    for persona in grupo_personas:
        df = persona.get_persona_df()
        if df is not None and len(df) > 0:
            # Obtener todos los valores ordenados temporalmente
            df_ordenado = df.sort_values(['recorte', 'indice'])
            valores = df_ordenado['valor'].tolist()
            
            sd1, sd2 = calcular_sd1_sd2(valores)
            
            if sd1 is not None and sd2 is not None:
                metricas.append({
                    'persona': persona.nombre,
                    'status': persona.status,
                    'sd1': sd1,
                    'sd2': sd2
                })
    
    return metricas

In [12]:
def leave_one_subject_out_knn(metricas_control, metricas_dcl, k=3):
    """
    Validación leave-one-subject-out usando tu KNN manual
    
    Parámetros:
    - metricas_control: lista de diccionarios con métricas de controles
    - metricas_dcl: lista de diccionarios con métricas de DCL
    - k: número de vecinos para KNN
    
    Retorna:
    - resultados: diccionario con predicciones y precisión
    """
    # Combinar todos los sujetos
    todos_sujetos = metricas_control + metricas_dcl
    
    resultados = {
        'sujeto': [],
        'real': [],
        'predicho': [],
        'correcto': []
    }
    
    print(f"{'='*60}")
    print(f"VALIDACIÓN LEAVE-ONE-SUBJECT-OUT")
    print(f"KNN con k={k}, distancia euclidiana")
    print(f"Total sujetos: {len(todos_sujetos)}")
    print(f"{'='*60}\n")
    
    for i, sujeto_test in enumerate(todos_sujetos):
        # Preparar datos de entrenamiento (todos excepto el sujeto actual)
        X_train = []
        y_train = []
        
        for j, sujeto in enumerate(todos_sujetos):
            if i != j:  # No incluir el sujeto de prueba
                X_train.append([sujeto['sd1'], sujeto['sd2']])
                y_train.append(sujeto['status'])
        
        # Preparar datos de prueba (solo el sujeto actual)
        X_test = [[sujeto_test['sd1'], sujeto_test['sd2']]]
        y_test = [sujeto_test['status']]
        
        # Usar tu función knn_predict
        predicciones = knn_predict(X_train, y_train, X_test, k=k)
        predicho = predicciones[0]
        real = y_test[0]
        
        # Guardar resultado
        correcto = predicho == real
        resultados['sujeto'].append(sujeto_test['persona'])
        resultados['real'].append(real)
        resultados['predicho'].append(predicho)
        resultados['correcto'].append(correcto)
        
        # Mostrar resultado
        real_texto = "DCL" if real == 1 else "Control"
        pred_texto = "DCL" if predicho == 1 else "Control"
        check = "✓" if correcto else "✗"
        print(f"{check} {sujeto_test['persona']:6} | Real: {real_texto:7} | Pred: {pred_texto:7}")
    
    # Estadísticas
    print(f"\n{'='*60}")
    print(f"RESULTADOS GLOBALES")
    print(f"{'='*60}")
    aciertos = sum(resultados['correcto'])
    total = len(resultados['correcto'])
    precision = aciertos / total
    print(f"Precisión total: {precision:.3f} ({aciertos}/{total})")
    
    # Matriz de confusión manual
    tp = sum(1 for i in range(total) if resultados['real'][i] == 1 and resultados['predicho'][i] == 1)
    tn = sum(1 for i in range(total) if resultados['real'][i] == 0 and resultados['predicho'][i] == 0)
    fp = sum(1 for i in range(total) if resultados['real'][i] == 0 and resultados['predicho'][i] == 1)
    fn = sum(1 for i in range(total) if resultados['real'][i] == 1 and resultados['predicho'][i] == 0)
    
    print(f"\nMatriz de confusión:")
    print(f"              Predicho")
    print(f"              Control  DCL")
    print(f"Real  Control    {tn:3}     {fp:3}")
    print(f"      DCL        {fn:3}     {tp:3}")
    
    if (tp + fp) > 0:
        precision_dcl = tp / (tp + fp) if (tp + fp) > 0 else 0
        print(f"\nPrecisión DCL: {precision_dcl:.3f}")
    if (tp + fn) > 0:
        recall_dcl = tp / (tp + fn) if (tp + fn) > 0 else 0
        print(f"Recall DCL: {recall_dcl:.3f}")
    
    return resultados

In [34]:
def preparar_metricas_desde_csv(sd1_df, sd2_df, status):
    """
    Prepara las métricas desde los DataFrames pivotados de sd1 y sd2
    
    Parámetros:
    - sd1_df: DataFrame con columnas 'persona' y nodos (valores sd1)
    - sd2_df: DataFrame con columnas 'persona' y nodos (valores sd2)
    - status: 0=control, 1=DCL
    
    Retorna:
    - Lista de diccionarios con persona, status, sd1 y sd2 (promedio por sujeto)
    """
    metricas = []
    
    for idx in range(len(sd1_df)):
        persona = sd1_df.iloc[idx]['persona']
        
        # Extraer todos los valores sd1 y sd2 de los nodos
        # Excluir la columna 'persona' y 'Unnamed: 0' si existe
        cols_nodos = [col for col in sd1_df.columns if col not in ['persona', 'Unnamed: 0']]
        
        # Obtener todos los valores sd1 y sd2 como arrays
        sd1_values = sd1_df.iloc[idx][cols_nodos].values.astype(float)
        sd2_values = sd2_df.iloc[idx][cols_nodos].values.astype(float)
        
        # Calcular la media de sd1 y sd2 por sujeto (o puedes usar el promedio de todos los nodos)
        sd1_mean = np.nanmean(sd1_values)
        sd2_mean = np.nanmean(sd2_values)
        
        metricas.append({
            'persona': persona,
            'status': status,
            'sd1': sd1_mean,
            'sd2': sd2_mean,
            # También guardar todos los nodos por si quieres usarlos como features multivariate
            'sd1_por_nodo': sd1_values,
            'sd2_por_nodo': sd2_values
        })
    
    return metricas

In [44]:
def leave_one_subject_out_knn_desde_csv(metricas_control, metricas_dcl, usar_todos_nodos=False, k=3):
    """
    Validación leave-one-subject-out usando los datos de los CSV
    
    Parámetros:
    - metricas_control: lista de métricas de controles
    - metricas_dcl: lista de métricas de DCL
    - usar_todos_nodos: si es True, usa todos los nodos como features (42 features: 21 sd1 + 21 sd2)
                        si es False, usa solo la media de sd1 y sd2 (2 features)
    - k: número de vecinos
    """
    # Combinar todos los sujetos
    todos_sujetos = metricas_control + metricas_dcl
    
    resultados = {
        'sujeto': [],
        'real': [],
        'predicho': [],
        'correcto': []
    }
    
    print(f"{'='*70}")
    print(f"VALIDACIÓN LEAVE-ONE-SUBJECT-OUT")
    print(f"Modo: {'Todos los nodos (42 features)' if usar_todos_nodos else 'Media SD1/SD2 (2 features)'}")
    print(f"KNN con k={k}, distancia euclidiana")
    print(f"Total sujetos: {len(todos_sujetos)}")
    print(f"{'='*70}\n")
    
    for i, sujeto_test in enumerate(todos_sujetos):
        # Preparar datos de entrenamiento
        X_train = []
        y_train = []
        
        for j, sujeto in enumerate(todos_sujetos):
            if i != j:  # Excluir sujeto de prueba
                if usar_todos_nodos:
                    # Concatenar sd1 y sd2 de todos los nodos
                    features = np.concatenate([sujeto['sd1_por_nodo'], sujeto['sd2_por_nodo']])
                    # Eliminar NaNs si existen
                    features = features[~np.isnan(features)]
                else:
                    # Usar solo la media de sd1 y sd2
                    features = [sujeto['sd1'], sujeto['sd2']]
                
                X_train.append(features)
                y_train.append(sujeto['status'])
        
        # Preparar datos de prueba
        if usar_todos_nodos:
            test_features = np.concatenate([sujeto_test['sd1_por_nodo'], sujeto_test['sd2_por_nodo']])
            test_features = test_features[~np.isnan(test_features)]
        else:
            test_features = [sujeto_test['sd1'], sujeto_test['sd2']]
        
        X_test = [test_features]
        real = sujeto_test['status']
        
        # Predecir
        predicciones = knn_predict(X_train, y_train, X_test, k=k)
        predicho = predicciones[0]
        
        # Guardar resultado
        correcto = predicho == real
        resultados['sujeto'].append(sujeto_test['persona'])
        resultados['real'].append(real)
        resultados['predicho'].append(predicho)
        resultados['correcto'].append(correcto)
        
        # Mostrar resultado
        real_texto = "DCL" if real == 1 else "Control"
        pred_texto = "DCL" if predicho == 1 else "Control"
        check = "✓" if correcto else "✗"
        print(f"{check} {sujeto_test['persona']:6} | Real: {real_texto:7} | Pred: {pred_texto:7}")
    
    # Estadísticas
    print(f"\n{'='*70}")
    print(f"RESULTADOS GLOBALES")
    print(f"{'='*70}")
    aciertos = sum(resultados['correcto'])
    total = len(resultados['correcto'])
    precision = aciertos / total
    print(f"Precisión total: {precision:.3f} ({aciertos}/{total})")
    
    # Matriz de confusión
    tp = sum(1 for i in range(total) if resultados['real'][i] == 1 and resultados['predicho'][i] == 1)
    tn = sum(1 for i in range(total) if resultados['real'][i] == 0 and resultados['predicho'][i] == 0)
    fp = sum(1 for i in range(total) if resultados['real'][i] == 0 and resultados['predicho'][i] == 1)
    fn = sum(1 for i in range(total) if resultados['real'][i] == 1 and resultados['predicho'][i] == 0)
    
    print(f"\nMatriz de confusión:")
    print(f"              Predicho")
    print(f"              Control  DCL")
    print(f"Real  Control    {tn:3}     {fp:3}")
    print(f"      DCL        {fn:3}     {tp:3}")
    
    if (tp + fp) > 0:
        precision_dcl = tp / (tp + fp)
        print(f"\nPrecisión DCL: {precision_dcl:.3f}")
    if (tp + fn) > 0:
        recall_dcl = tp / (tp + fn)
        print(f"Recall DCL: {recall_dcl:.3f}")
    
    if (tp + fp) > 0 and (tp + fn) > 0:
        f1_dcl = 2 * (precision_dcl * recall_dcl) / (precision_dcl + recall_dcl)
        print(f"F1-Score DCL: {f1_dcl:.3f}")
    
    return resultados

In [53]:
def leave_one_subject_out_knn_por_nodo(metricas_control, metricas_dcl, k=3):
    """
    Valida cada nodo individualmente y combina por votación
    """
    # Obtener nombres de nodos
    nodos = ['C3','C4','CZ','F3','F4','F7','F8','FP1','FP2','FZ','LOG',
             'O1','O2','P3','P4','PZ','ROG','T3','T4','T5','T6']
    
    todos_sujetos = metricas_control + metricas_dcl
    votaciones = {sujeto['persona']: {'control': 0, 'dcl': 0} for sujeto in todos_sujetos}
    
    print(f"{'='*70}")
    print(f"VALIDACIÓN LEAVE-ONE-SUBJECT-OUT POR NODO")
    print(f"KNN con k={k}, distancia euclidiana")
    print(f"Total sujetos: {len(todos_sujetos)}")
    print(f"Total nodos: {len(nodos)}")
    print(f"{'='*70}\n")
    
    for nodo_idx, nodo in enumerate(nodos):
        # Preparar datos para este nodo
        datos_nodo = []
        for sujeto in todos_sujetos:
            sd1 = sujeto['sd1_por_nodo'][nodo_idx]
            sd2 = sujeto['sd2_por_nodo'][nodo_idx]
            if not np.isnan(sd1) and not np.isnan(sd2):
                datos_nodo.append({
                    'persona': sujeto['persona'],
                    'status': sujeto['status'],
                    'features': [sd1, sd2]
                })
        
        # Leave-one-out para este nodo
        for i, test_sujeto in enumerate(datos_nodo):
            X_train = []
            y_train = []
            for j, train_sujeto in enumerate(datos_nodo):
                if i != j:
                    X_train.append(train_sujeto['features'])
                    y_train.append(train_sujeto['status'])
            
            X_test = [test_sujeto['features']]
            pred = knn_predict(X_train, y_train, X_test, k=k)[0]
            
            if pred == 1:
                votaciones[test_sujeto['persona']]['dcl'] += 1
            else:
                votaciones[test_sujeto['persona']]['control'] += 1
    
    # Decisión final por mayoría
    print(f"{'='*70}")
    print(f"RESULTADOS POR VOTACIÓN MAYORITARIA")
    print(f"{'='*70}")
    
    aciertos = 0
    for persona, votos in votaciones.items():
        # Encontrar status real
        real = next(s for s in todos_sujetos if s['persona'] == persona)['status']
        
        if votos['dcl'] > votos['control']:
            predicho = 1
        else:
            predicho = 0
        
        correcto = predicho == real
        if correcto:
            aciertos += 1
        
        real_texto = "DCL" if real == 1 else "Control"
        pred_texto = "DCL" if predicho == 1 else "Control"
        check = "✓" if correcto else "✗"
        print(f"{check} {persona:6} | Real: {real_texto:7} | Pred: {pred_texto:7} | "
              f"Votos: C={votos['control']}, D={votos['dcl']}")
    
    precision = aciertos / len(todos_sujetos)
    print(f"\nPrecisión total: {precision:.3f} ({aciertos}/{len(todos_sujetos)})")
    
    return votaciones

In [56]:
def leave_one_subject_out_rf(metricas_control, metricas_dcl, usar_todos_nodos=False, n_estimators=100, max_depth=None, random_state=42):
    """
    Validación leave-one-subject-out con Random Forest
    
    Parámetros:
    - metricas_control: lista de métricas de controles
    - metricas_dcl: lista de métricas de DCL
    - usar_todos_nodos: si es True, usa todos los nodos como features
    - n_estimators: número de árboles en el bosque
    - max_depth: profundidad máxima de los árboles
    - random_state: semilla para reproducibilidad
    """
    # Combinar todos los sujetos
    todos_sujetos = metricas_control + metricas_dcl
    
    resultados = {
        'sujeto': [],
        'real': [],
        'predicho': [],
        'probabilidades': [],
        'correcto': []
    }
    
    print(f"{'='*70}")
    print(f"VALIDACIÓN LEAVE-ONE-SUBJECT-OUT - RANDOM FOREST")
    print(f"Modo: {'Todos los nodos' if usar_todos_nodos else 'Media SD1/SD2'}")
    print(f"n_estimators: {n_estimators}, max_depth: {max_depth if max_depth else 'None'}")
    print(f"Total sujetos: {len(todos_sujetos)}")
    print(f"{'='*70}\n")
    
    for i, sujeto_test in enumerate(todos_sujetos):
        # Preparar datos de entrenamiento
        X_train = []
        y_train = []
        
        for j, sujeto in enumerate(todos_sujetos):
            if i != j:  # Excluir sujeto de prueba
                if usar_todos_nodos:
                    features = np.concatenate([sujeto['sd1_por_nodo'], sujeto['sd2_por_nodo']])
                    features = np.nan_to_num(features, nan=0.0)
                else:
                    features = np.array([sujeto['sd1'], sujeto['sd2']])
                
                X_train.append(features)
                y_train.append(sujeto['status'])
        
        # Preparar datos de prueba
        if usar_todos_nodos:
            test_features = np.concatenate([sujeto_test['sd1_por_nodo'], sujeto_test['sd2_por_nodo']])
            test_features = np.nan_to_num(test_features, nan=0.0)
        else:
            test_features = np.array([sujeto_test['sd1'], sujeto_test['sd2']])
        
        X_train = np.array(X_train)
        X_test = test_features.reshape(1, -1)
        real = sujeto_test['status']
        
        # Entrenar Random Forest
        rf = RandomForestClassifier(n_estimators=n_estimators, 
                                    max_depth=max_depth,
                                    random_state=random_state,
                                    n_jobs=-1)
        rf.fit(X_train, y_train)
        
        # Predecir
        predicho = rf.predict(X_test)[0]
        probabilidades = rf.predict_proba(X_test)[0]
        
        # Guardar resultado
        correcto = predicho == real
        resultados['sujeto'].append(sujeto_test['persona'])
        resultados['real'].append(real)
        resultados['predicho'].append(predicho)
        resultados['probabilidades'].append(probabilidades)
        resultados['correcto'].append(correcto)
        
        # Mostrar resultado
        real_texto = "DCL" if real == 1 else "Control"
        pred_texto = "DCL" if predicho == 1 else "Control"
        prob_dcl = probabilidades[1] if len(probabilidades) > 1 else 0
        check = "✓" if correcto else "✗"
        print(f"{check} {sujeto_test['persona']:6} | Real: {real_texto:7} | Pred: {pred_texto:7} | Prob DCL: {prob_dcl:.3f}")
    
    # Estadísticas
    print(f"\n{'='*70}")
    print(f"RESULTADOS GLOBALES - RANDOM FOREST")
    print(f"{'='*70}")
    aciertos = sum(resultados['correcto'])
    total = len(resultados['correcto'])
    precision = aciertos / total
    print(f"Precisión total: {precision:.3f} ({aciertos}/{total})")
    
    # Matriz de confusión
    tp = sum(1 for i in range(total) if resultados['real'][i] == 1 and resultados['predicho'][i] == 1)
    tn = sum(1 for i in range(total) if resultados['real'][i] == 0 and resultados['predicho'][i] == 0)
    fp = sum(1 for i in range(total) if resultados['real'][i] == 0 and resultados['predicho'][i] == 1)
    fn = sum(1 for i in range(total) if resultados['real'][i] == 1 and resultados['predicho'][i] == 0)
    
    print(f"\nMatriz de confusión:")
    print(f"              Predicho")
    print(f"              Control  DCL")
    print(f"Real  Control    {tn:3}     {fp:3}")
    print(f"      DCL        {fn:3}     {tp:3}")
    
    if (tp + fp) > 0:
        precision_dcl = tp / (tp + fp)
        print(f"\nPrecisión DCL: {precision_dcl:.3f}")
    if (tp + fn) > 0:
        recall_dcl = tp / (tp + fn)
        print(f"Recall DCL: {recall_dcl:.3f}")
    
    if (tp + fp) > 0 and (tp + fn) > 0:
        f1_dcl = 2 * (precision_dcl * recall_dcl) / (precision_dcl + recall_dcl)
        print(f"F1-Score DCL: {f1_dcl:.3f}")
    
    return resultados

In [13]:

ruta_base_DCL = "/home/jesusmendoza/Clasificadores/Recortes_aleatorios/DCL/"
ruta_base_Control = "/home/jesusmendoza/Clasificadores/Recortes_aleatorios/Control/"

# Nodos
nodos = ["C3","C4","CZ", "F3","F4","F7","F8","FP1","FP2","FZ","LOG","O1","O2","P3", "P4", "PZ", "ROG", "T3", "T4", "T5", "T6"]

# Nombres de los sujetos
nombresDCL = ['AEF', 'CLM', 'FGV', 'JGM', 'LIV', 'PCM', 'RLM', 'RRM']
nombresControl = ['EMV', 'GH2', 'GUR', 'JAL', 'JAN', 'MGN', 'MJN', 'MMA', 'RAN', 'VCN']

In [8]:
#pruebaa
# Prueba con un solo sujeto
prueba = Persona(ruta_base_Control, nombresControl[0], nodos, 0)
prueba.all_files()
print(prueba.data.head())
print(f"\nForma del DataFrame: {prueba.data.shape}")
print(f"\nEstadísticas básicas:")
print(prueba.data['valor'].describe())

✓ EMV: 8064000 registros cargados
  nodo  recorte  indice  valor persona  status
0   C3        1       0    3.6     EMV       0
1   C3        1       1    2.8     EMV       0
2   C3        1       2    2.9     EMV       0
3   C3        1       3    3.1     EMV       0
4   C3        1       4    2.1     EMV       0

Forma del DataFrame: (8064000, 6)

Estadísticas básicas:
count    8.064000e+06
mean    -1.916264e-01
std      5.999171e+00
min     -6.050000e+01
25%     -3.100000e+00
50%     -4.000000e-01
75%      3.000000e+00
max      2.094000e+02
Name: valor, dtype: float64


In [9]:
personasControl = cargar_grupo(nombresControl, ruta_base_Control, nodos, 0)

Cargando EMV (1/10)...
✓ EMV: 8064000 registros cargados
Cargando GH2 (2/10)...
✓ GH2: 8064000 registros cargados
Cargando GUR (3/10)...
✓ GUR: 8064000 registros cargados
Cargando JAL (4/10)...
✓ JAL: 8064000 registros cargados
Cargando JAN (5/10)...
✓ JAN: 8064000 registros cargados
Cargando MGN (6/10)...
✓ MGN: 8064000 registros cargados
Cargando MJN (7/10)...
✓ MJN: 8064000 registros cargados
Cargando MMA (8/10)...
✓ MMA: 8064000 registros cargados
Cargando RAN (9/10)...
✓ RAN: 8064000 registros cargados
Cargando VCN (10/10)...
✓ VCN: 8064000 registros cargados


In [ ]:
print(personasControl.data['EMV'].describe())

In [48]:
df_control = estadistica_grupo(personasControl, "CONTROL")



ESTADÍSTICA DESCRIPTIVA - GRUPO CONTROL
Total de registros: 80640000
Personas: 10
Nodos: 21
Recortes: 50

--- Estadísticas generales del valor ---
count    8.064000e+07
mean    -1.210834e+00
std      4.451303e+00
min     -9.820000e+01
25%     -3.200000e+00
50%     -1.200000e+00
75%      8.000000e-01
max      2.094000e+02
Name: valor, dtype: float64

--- Estadísticas por persona ---
             mean       std    count
persona                             
EMV     -0.191626  5.999171  8064000
GH2      1.229458  3.242367  8064000
GUR     -0.982187  5.946566  8064000
JAL     -1.113644  3.809465  8064000
JAN     -0.731313  4.504231  8064000
MGN      0.279428  3.652756  8064000
MJN     -2.711948  4.614305  8064000
MMA     -2.633657  1.835768  8064000
RAN     -1.696090  4.123038  8064000
VCN     -3.556757  2.636170  8064000

--- Estadísticas por nodo ---
          mean       std
nodo                    
C3   -1.243891  3.341318
C4   -0.217517  3.158009
CZ    0.018206  3.210900
F3   -0.955585

In [27]:
sd1_sd2_control = calcular_metricas_poincare_por_nodo(personasControl)
sd1_sd2_control

,persona,nodo,status,sd1,sd2
0,EMV,C3,0,0.882621,2.662334
1,EMV,C4,0,0.650410,2.702103
2,EMV,CZ,0,0.530723,2.796299
3,EMV,F3,0,2.051115,3.962141
4,EMV,F4,0,1.411846,5.237217
...,...,...,...,...,...
205,VCN,ROG,0,0.153421,5.197812
206,VCN,T3,0,1.687976,1.878639
207,VCN,T4,0,0.741090,2.036916
208,VCN,T5,0,1.111960,1.780630


In [45]:
# Pivot para tener nodos como columnas
df_sd1_pivot = sd1_sd2_control.pivot(index='persona', columns='nodo', values='sd1')
df_sd2_pivot = sd1_sd2_control.pivot(index='persona', columns='nodo', values='sd2')

df_sd1_pivot.to_csv('sd1_Control')
df_sd2_pivot.to_csv('sd2_Control')


SD1 por persona y nodo:


In [37]:
sd1_Control = pd.read_csv('sd1_Control')
sd2_Control = pd.read_csv('sd2_Control')

In [47]:
sd1_Control

,persona,C3,C4,CZ,F3,F4,F7,F8,FP1,FP2,...,O1,O2,P3,P4,PZ,ROG,T3,T4,T5,T6
0,EMV,0.882621,0.650410,0.530723,2.051115,1.411846,0.947448,1.013690,1.007131,0.706254,...,0.384277,0.382179,0.481514,0.519768,0.449756,0.147840,0.688849,1.081821,0.511615,0.414849
1,GH2,1.125456,0.794126,0.620437,1.281317,0.853582,0.804547,0.563329,0.689054,0.747789,...,0.501197,0.559233,0.689113,0.510861,0.520456,0.129643,1.007952,1.055303,0.733965,0.588127
2,GUR,0.811069,0.728736,0.588916,1.297132,1.039410,0.605073,0.637269,0.849251,0.974940,...,0.422371,0.436826,0.453867,0.509529,0.481995,0.203235,0.983095,0.748063,0.461299,0.469776
3,JAL,0.172406,0.176044,0.152539,0.191704,0.172210,0.183805,0.192727,0.218725,0.284348,...,0.172096,0.156669,0.149101,0.149144,0.154435,0.065993,0.143250,0.147078,0.158468,0.142090
4,JAN,0.284314,0.567027,0.276553,0.323819,0.857937,0.321142,0.603695,0.400554,0.408922,...,0.258640,0.295602,0.266175,0.285215,0.264762,0.124850,0.364336,0.510399,0.289686,0.317153
5,MGN,0.163050,0.154986,0.155906,0.235969,0.301576,0.232409,0.252066,0.321750,0.308264,...,0.172028,0.208148,0.151309,0.164606,0.160214,0.079442,0.160840,0.149320,0.126883,0.162372
6,MJN,0.227513,0.387376,0.192483,0.375150,0.305000,0.341603,0.405013,0.358215,0.267050,...,0.187214,0.224776,0.199654,0.237192,0.188721,0.085668,0.361329,0.575365,0.312354,0.398467
7,MMA,0.337479,0.327070,0.223870,0.337422,0.386074,0.312112,0.309756,0.294791,0.303024,...,0.201340,0.202690,0.209681,0.211025,0.208345,0.075928,0.484477,0.300150,0.262046,0.308913
8,RAN,0.660353,0.455465,0.393959,0.573779,0.658179,1.092891,0.698647,0.664337,0.842181,...,0.371324,0.368095,0.429184,0.386801,0.370813,0.160184,1.069402,0.883017,0.394281,0.423095
9,VCN,3.216339,0.669013,0.944499,3.613826,1.238651,2.523170,0.531924,1.395581,1.171437,...,0.674767,0.596703,1.220675,0.586792,0.729740,0.153421,1.687976,0.741090,1.111960,0.667617


In [51]:
sd2_Control

,persona,C3,C4,CZ,F3,F4,F7,F8,FP1,FP2,...,O1,O2,P3,P4,PZ,ROG,T3,T4,T5,T6
0,EMV,2.662334,2.702103,2.796299,3.962141,5.237217,3.312572,5.811778,12.645906,14.391724,...,2.065285,2.515431,2.572517,2.892038,2.477709,7.209660,2.620931,3.036861,2.049614,2.177038
1,GH2,2.931719,2.860748,2.768246,4.047732,4.257575,4.078308,4.061122,8.740269,8.132469,...,2.167325,2.251489,2.373592,2.500200,2.475266,3.496351,2.539790,2.642893,2.244875,2.250104
2,GUR,8.242728,8.231540,8.230525,8.630446,8.762660,8.058356,8.357935,12.167442,10.693582,...,7.768309,7.964356,7.850115,8.123482,7.976335,4.676628,7.577388,8.423524,7.838593,7.921391
3,JAL,2.206652,2.445058,2.760225,4.227327,4.820216,3.697312,4.136101,10.892473,10.985331,...,6.406035,1.798978,1.708626,2.168105,4.128154,3.907985,1.424068,1.613890,1.326364,1.609543
4,JAN,3.063320,3.207034,3.127735,4.014516,3.834376,6.219179,5.546359,6.690734,4.164247,...,2.819546,2.741228,2.862922,2.844857,2.817278,10.187249,3.218067,2.824215,2.595279,2.614180
5,MGN,3.188209,3.529403,2.950655,3.723031,6.788143,5.283628,5.377320,7.484065,6.585606,...,4.937005,5.657200,2.233225,3.592602,3.867739,4.933679,3.036477,3.045216,5.459718,3.862023
6,MJN,3.058315,2.782428,2.448952,4.181402,3.726168,4.139418,4.842484,7.227326,12.764450,...,2.755036,4.050511,2.177763,2.123588,3.294569,4.703229,4.252993,2.822532,3.132285,3.247139
7,MMA,1.708322,1.614103,1.800799,1.958009,2.033826,1.956030,2.289987,4.472758,4.355816,...,1.272827,1.239233,1.410219,1.463270,1.565019,1.971487,1.508476,1.404021,1.354713,1.356880
8,RAN,2.445634,3.283274,2.707315,3.328854,5.752879,3.969847,11.851277,10.179454,11.670784,...,2.252400,2.078084,2.029475,2.122360,2.139100,5.459521,3.676767,5.966474,1.978935,2.673462
9,VCN,2.789863,1.725497,1.704511,3.420785,2.110813,3.117290,2.453890,2.875829,2.808296,...,1.911828,2.809551,1.824361,1.745815,1.720656,5.197812,1.878639,2.036916,1.780630,3.764514


In [14]:
personasDCL = cargar_grupo(nombresDCL, ruta_base_DCL, nodos, 1)

Cargando AEF (1/8)...
✓ AEF: 8064000 registros cargados
Cargando CLM (2/8)...
✓ CLM: 8064000 registros cargados
Cargando FGV (3/8)...
✓ FGV: 8064000 registros cargados
Cargando JGM (4/8)...
✓ JGM: 8064000 registros cargados
Cargando LIV (5/8)...
✓ LIV: 8064000 registros cargados
Cargando PCM (6/8)...
✓ PCM: 8064000 registros cargados
Cargando RLM (7/8)...
✓ RLM: 8064000 registros cargados
Cargando RRM (8/8)...
✓ RRM: 8064000 registros cargados


In [15]:
df_dcl = estadistica_grupo(personasDCL, "DCL")


ESTADÍSTICA DESCRIPTIVA - GRUPO DCL
Total de registros: 64512000
Personas: 8
Nodos: 21
Recortes: 50

--- Estadísticas generales del valor ---
count    6.451200e+07
mean    -3.671997e+00
std      6.614017e+00
min     -1.319000e+02
25%     -6.900000e+00
50%     -2.200000e+00
75%      0.000000e+00
max      4.102000e+02
Name: valor, dtype: float64

--- Estadísticas por persona ---
              mean       std    count
persona                              
AEF      -1.489483  4.022626  8064000
CLM      -2.558768  4.074282  8064000
FGV      -1.888825  2.818142  8064000
JGM      -0.340215  5.504490  8064000
LIV      -9.827411  4.448651  8064000
PCM       1.656528  4.694538  8064000
RLM      -1.669623  6.255169  8064000
RRM     -13.258175  3.938663  8064000

--- Estadísticas por nodo ---
          mean       std
nodo                    
C3   -4.330977  5.966704
C4   -3.826624  5.741428
CZ   -3.187523  6.135972
F3   -3.930274  6.265975
F4   -3.530412  6.382223
F7   -3.674240  6.221246
F8   -3.

In [28]:
for i in range(8):
    print(personasDCL[i].data['valor'].describe())


count    8.064000e+06
mean    -1.489483e+00
std      4.022626e+00
min     -6.270000e+01
25%     -2.500000e+00
50%     -1.400000e+00
75%     -4.000000e-01
max      3.356000e+02
Name: valor, dtype: float64
count    8.064000e+06
mean    -2.558768e+00
std      4.074282e+00
min     -5.850000e+01
25%     -3.800000e+00
50%     -2.400000e+00
75%     -1.000000e+00
max      6.050000e+01
Name: valor, dtype: float64
count    8.064000e+06
mean    -1.888825e+00
std      2.818142e+00
min     -3.630000e+01
25%     -3.400000e+00
50%     -2.200000e+00
75%     -7.000000e-01
max      5.340000e+01
Name: valor, dtype: float64
count    8.064000e+06
mean    -3.402148e-01
std      5.504490e+00
min     -4.590000e+01
25%     -2.500000e+00
50%     -5.000000e-01
75%      1.500000e+00
max      9.640000e+01
Name: valor, dtype: float64
count    8.064000e+06
mean    -9.827411e+00
std      4.448651e+00
min     -5.120000e+01
25%     -1.150000e+01
50%     -1.030000e+01
75%     -8.600000e+00
max      3.890000e+01
Name: va

In [30]:
sd1_sd2_DCL = calcular_metricas_poincare_por_nodo(personasDCL)
sd1_sd2_DCL

,persona,nodo,status,sd1,sd2
0,AEF,C3,1,0.234897,2.020237
1,AEF,C4,1,0.288254,1.637607
2,AEF,CZ,1,0.215284,1.852967
3,AEF,F3,1,0.326326,3.217543
4,AEF,F4,1,0.407670,3.149068
...,...,...,...,...,...
163,RRM,ROG,1,0.310506,9.267685
164,RRM,T3,1,0.893770,2.043539
165,RRM,T4,1,1.197106,1.885349
166,RRM,T5,1,1.280075,1.556479


In [31]:
# Pivot para tener nodos como columnas
df_sd1_pivot_dcl = sd1_sd2_DCL.pivot(index='persona', columns='nodo', values='sd1')
df_sd2_pivot_dcl = sd1_sd2_DCL.pivot(index='persona', columns='nodo', values='sd2')

df_sd1_pivot_dcl.to_csv('sd1_DCL')
df_sd2_pivot_dcl.to_csv('sd2_DCL')

SD1 por persona y nodo:


In [48]:
sd1_DCL = pd.read_csv('sd1_DCL')
sd2_DCL = pd.read_csv('sd2_DCL')

In [49]:
sd1_DCL

,persona,C3,C4,CZ,F3,F4,F7,F8,FP1,FP2,...,O1,O2,P3,P4,PZ,ROG,T3,T4,T5,T6
0,AEF,0.234897,0.288254,0.215284,0.326326,0.407670,0.484793,0.490376,0.521727,0.452218,...,0.221873,0.230344,0.227768,0.246919,0.208833,0.070539,0.300487,0.486410,0.293712,0.321065
1,CLM,0.288738,0.305976,0.241438,0.283485,0.289710,0.257632,0.362622,0.283827,0.329851,...,0.284207,0.460702,0.340711,0.299212,0.241523,0.100955,0.410219,0.380871,0.500926,0.378765
2,FGV,0.379804,0.379690,0.347081,0.509608,0.705338,0.499642,0.460538,0.676633,1.154031,...,0.463173,0.839172,0.368522,0.428597,0.387316,0.145397,0.466827,0.404735,0.408362,0.545414
3,JGM,0.767247,0.930426,0.483763,1.003160,1.234458,0.549936,0.684066,0.586202,0.556219,...,0.379385,0.536733,0.478348,0.646581,0.438448,0.129210,0.675884,0.876458,0.886659,0.715850
4,LIV,0.317291,0.377727,0.336321,0.406583,0.392730,0.363697,0.300516,0.615090,0.613488,...,0.388492,0.476393,0.290725,0.316089,0.297558,0.101047,0.387188,0.295153,0.321733,0.399954
5,PCM,1.059118,0.939789,0.657271,1.647116,1.575456,1.178124,1.048115,0.913224,0.893278,...,0.534457,0.475162,0.551757,0.604069,0.510018,0.076560,0.756063,0.581946,0.530353,0.536027
6,RLM,0.219800,0.224618,0.192925,0.329084,0.388618,0.225485,0.295913,0.580185,0.500511,...,0.224140,0.223962,0.218026,0.207523,0.183471,0.105648,0.269391,0.245191,0.327620,0.387181
7,RRM,0.794584,0.818349,0.600895,0.781611,0.717694,0.935557,0.852683,1.209655,1.019444,...,0.890594,0.901891,0.701873,0.899669,0.611362,0.310506,0.893770,1.197106,1.280075,1.300868


In [50]:
sd2_DCL

,persona,C3,C4,CZ,F3,F4,F7,F8,FP1,FP2,...,O1,O2,P3,P4,PZ,ROG,T3,T4,T5,T6
0,AEF,2.020237,1.637607,1.852967,3.217543,3.149068,2.902293,2.913621,19.899625,9.060672,...,1.342155,1.252818,1.503818,1.480187,1.471400,2.496012,2.441676,1.551497,1.440219,1.256695
1,CLM,2.499746,2.955872,2.777454,3.625727,4.395817,4.362392,4.923512,9.598879,12.497604,...,10.932876,4.249054,1.928453,2.405467,2.207952,7.295341,2.690491,4.506661,2.432573,2.133910
2,FGV,2.319008,1.810343,2.158681,3.338242,3.005729,4.214698,2.487429,8.280130,6.843299,...,1.733082,2.051266,1.588805,1.682060,1.742327,3.177008,2.080206,1.689378,1.341709,1.384077
3,JGM,5.487168,4.790052,4.349693,5.994039,5.881203,4.644303,5.685987,11.559928,11.658969,...,6.188964,13.927148,3.822568,7.022102,15.089886,6.482856,3.574042,4.914064,3.480144,4.218960
4,LIV,5.311076,4.849396,4.979661,5.908914,6.085140,6.810907,5.281613,8.731965,10.369988,...,4.950243,4.500427,4.827435,4.869221,4.732319,1.782934,5.329115,4.856990,4.993527,4.459967
5,PCM,4.888106,4.906231,3.928573,6.937266,7.249002,4.857967,6.365813,11.363102,15.000924,...,2.677858,2.828534,3.270717,3.682619,3.370678,3.882606,4.379295,4.518036,2.683619,3.114656
6,RLM,5.270707,5.099075,5.250865,6.256892,5.831902,6.540363,5.725339,10.189019,14.028556,...,7.124400,8.841441,5.058507,5.745954,5.472678,14.025809,5.132804,5.273846,4.873631,5.088517
7,RRM,2.049859,2.360034,2.062717,2.960580,2.826745,3.848081,3.537034,6.810312,6.118349,...,1.467318,1.404167,1.676487,1.990203,1.784583,9.267685,2.043539,1.885349,1.556479,1.607860


In [41]:
# Prepara las métricas para controles y DCL
metricas_control = preparar_metricas_desde_csv(sd1_Control, sd2_Control, 0)
metricas_dcl = preparar_metricas_desde_csv(sd1_DCL, sd2_DCL, 1)

print(f"Controles cargados: {len(metricas_control)}")
print(f"DCL cargados: {len(metricas_dcl)}")

Controles cargados: 10
DCL cargados: 8


In [46]:
resultados_2f = leave_one_subject_out_knn_desde_csv(
    metricas_control, metricas_dcl, 
    usar_todos_nodos=False, 
    k=3
)


VALIDACIÓN LEAVE-ONE-SUBJECT-OUT
Modo: Media SD1/SD2 (2 features)
KNN con k=3, distancia euclidiana
Total sujetos: 18

✗ EMV    | Real: Control | Pred: DCL    
✗ GH2    | Real: Control | Pred: DCL    
✗ GUR    | Real: Control | Pred: DCL    
✓ JAL    | Real: Control | Pred: Control
✓ JAN    | Real: Control | Pred: Control
✓ MGN    | Real: Control | Pred: Control
✓ MJN    | Real: Control | Pred: Control
✗ MMA    | Real: Control | Pred: DCL    
✓ RAN    | Real: Control | Pred: Control
✗ VCN    | Real: Control | Pred: DCL    
✓ AEF    | Real: DCL     | Pred: DCL    
✗ CLM    | Real: DCL     | Pred: Control
✓ FGV    | Real: DCL     | Pred: DCL    
✓ JGM    | Real: DCL     | Pred: DCL    
✓ LIV    | Real: DCL     | Pred: DCL    
✓ PCM    | Real: DCL     | Pred: DCL    
✓ RLM    | Real: DCL     | Pred: DCL    
✓ RRM    | Real: DCL     | Pred: DCL    

RESULTADOS GLOBALES
Precisión total: 0.667 (12/18)

Matriz de confusión:
              Predicho
              Control  DCL
Real  Control      

In [54]:
resultados_42f = leave_one_subject_out_knn_por_nodo(
    metricas_control, metricas_dcl,  
    k=3
)

VALIDACIÓN LEAVE-ONE-SUBJECT-OUT POR NODO
KNN con k=3, distancia euclidiana
Total sujetos: 18
Total nodos: 21

RESULTADOS POR VOTACIÓN MAYORITARIA
✓ EMV    | Real: Control | Pred: Control | Votos: C=14, D=7
✓ GH2    | Real: Control | Pred: Control | Votos: C=16, D=5
✗ GUR    | Real: Control | Pred: DCL     | Votos: C=3, D=18
✗ JAL    | Real: Control | Pred: DCL     | Votos: C=8, D=13
✓ JAN    | Real: Control | Pred: Control | Votos: C=14, D=7
✗ MGN    | Real: Control | Pred: DCL     | Votos: C=10, D=11
✓ MJN    | Real: Control | Pred: Control | Votos: C=14, D=7
✗ MMA    | Real: Control | Pred: DCL     | Votos: C=3, D=18
✓ RAN    | Real: Control | Pred: Control | Votos: C=13, D=8
✓ VCN    | Real: Control | Pred: Control | Votos: C=12, D=9
✗ AEF    | Real: DCL     | Pred: Control | Votos: C=13, D=8
✗ CLM    | Real: DCL     | Pred: Control | Votos: C=17, D=4
✗ FGV    | Real: DCL     | Pred: Control | Votos: C=13, D=8
✗ JGM    | Real: DCL     | Pred: Control | Votos: C=11, D=10
✓ LIV    | 

In [60]:
resultados_42f_rf = leave_one_subject_out_rf(
    metricas_control, metricas_dcl,
    usar_todos_nodos = True
)

VALIDACIÓN LEAVE-ONE-SUBJECT-OUT - RANDOM FOREST
Modo: Todos los nodos
n_estimators: 100, max_depth: None
Total sujetos: 18

✓ EMV    | Real: Control | Pred: Control | Prob DCL: 0.440
✗ GH2    | Real: Control | Pred: DCL     | Prob DCL: 0.510
✗ GUR    | Real: Control | Pred: DCL     | Prob DCL: 0.750
✓ JAL    | Real: Control | Pred: Control | Prob DCL: 0.420
✗ JAN    | Real: Control | Pred: DCL     | Prob DCL: 0.540
✓ MGN    | Real: Control | Pred: Control | Prob DCL: 0.450
✓ MJN    | Real: Control | Pred: Control | Prob DCL: 0.400
✗ MMA    | Real: Control | Pred: DCL     | Prob DCL: 0.620
✗ RAN    | Real: Control | Pred: DCL     | Prob DCL: 0.520
✓ VCN    | Real: Control | Pred: Control | Prob DCL: 0.470
✗ AEF    | Real: DCL     | Pred: Control | Prob DCL: 0.250
✗ CLM    | Real: DCL     | Pred: Control | Prob DCL: 0.310
✗ FGV    | Real: DCL     | Pred: Control | Prob DCL: 0.400
✓ JGM    | Real: DCL     | Pred: DCL     | Prob DCL: 0.580
✓ LIV    | Real: DCL     | Pred: DCL     | Prob D

In [62]:
resultados_2f_rf = leave_one_subject_out_rf(
    metricas_control, metricas_dcl
)

VALIDACIÓN LEAVE-ONE-SUBJECT-OUT - RANDOM FOREST
Modo: Media SD1/SD2
n_estimators: 100, max_depth: None
Total sujetos: 18

✓ EMV    | Real: Control | Pred: Control | Prob DCL: 0.480
✗ GH2    | Real: Control | Pred: DCL     | Prob DCL: 0.600
✗ GUR    | Real: Control | Pred: DCL     | Prob DCL: 0.880
✓ JAL    | Real: Control | Pred: Control | Prob DCL: 0.050
✓ JAN    | Real: Control | Pred: Control | Prob DCL: 0.440
✓ MGN    | Real: Control | Pred: Control | Prob DCL: 0.310
✓ MJN    | Real: Control | Pred: Control | Prob DCL: 0.330
✗ MMA    | Real: Control | Pred: DCL     | Prob DCL: 0.530
✓ RAN    | Real: Control | Pred: Control | Prob DCL: 0.460
✗ VCN    | Real: Control | Pred: DCL     | Prob DCL: 0.790
✗ AEF    | Real: DCL     | Pred: Control | Prob DCL: 0.410
✗ CLM    | Real: DCL     | Pred: Control | Prob DCL: 0.220
✗ FGV    | Real: DCL     | Pred: Control | Prob DCL: 0.280
✗ JGM    | Real: DCL     | Pred: Control | Prob DCL: 0.420
✓ LIV    | Real: DCL     | Pred: DCL     | Prob DCL